In [6]:
import numpy as np
from scipy.stats import pearsonr
from tqdm import tqdm

def calc_r_rmse_maps_like_old(predicty, testy, show_progress=False):
    """
    完全按照你原来的思路：
    - 对每个格点 (i,j)，沿 time 计算 pearsonr
    - 对每个格点 (i,j)，沿 time 计算 RMSE

    输入
    ----
    predicty, testy : np.ndarray
        shape = (time, lat, lon)

    输出
    ----
    r_map, rmse_map : np.ndarray
        shape = (lat, lon)
    """
    predicty = np.asarray(predicty, dtype=np.float64)
    testy = np.asarray(testy, dtype=np.float64)

    if predicty.shape != testy.shape:
        raise ValueError(f"shape 不一致: {predicty.shape} vs {testy.shape}")
    if predicty.ndim != 3:
        raise ValueError("输入必须是三维 (time, lat, lon)")

    _, nlat, nlon = predicty.shape
    r_map = np.full((nlat, nlon), np.nan, dtype=np.float64)
    rmse_map = np.full((nlat, nlon), np.nan, dtype=np.float64)

    lat_iter = range(nlat)
    if show_progress:
        lat_iter = tqdm(lat_iter)

    for i in lat_iter:
        for j in range(nlon):
            x = predicty[:, i, j]
            y = testy[:, i, j]

            # 为了和你原来的结果尽量对齐：
            # 只要该格点时间序列中存在 NaN，就直接给 NaN
            if np.isnan(x).any() or np.isnan(y).any():
                r_map[i, j] = np.nan
                rmse_map[i, j] = np.nan
                continue

            # R
            try:
                r_map[i, j], _ = pearsonr(x, y)
            except:
                r_map[i, j] = np.nan

            # RMSE
            try:
                rmse_map[i, j] = np.sqrt(np.mean((x - y) ** 2))
            except:
                rmse_map[i, j] = np.nan

    return r_map, rmse_map

import numpy as np
from tqdm import tqdm

def _sample_block_indices(T, block_size, rng):
    """
    moving block bootstrap:
    每次随机抽连续块，直到拼满长度 T
    """
    if block_size < 1:
        raise ValueError("block_size 必须 >= 1")
    block_size = min(block_size, T)

    idx = []
    max_start = T - block_size

    while len(idx) < T:
        start = 0 if max_start <= 0 else rng.integers(0, max_start + 1)
        idx.extend(range(start, start + block_size))

    return np.array(idx[:T], dtype=int)


def bootstrap_ci_like_old_r_rmse(
    predicty,
    testy,
    n_boot=1000,
    block_size=5,
    ci=95,
    random_state=42,
    show_progress=True
):
    """
    按你原来的 R 定义做 block bootstrap 95% CI

    统计量定义：
    - R_overall    = nanmean(每个格点沿time算出来的R图)
    - RMSE_overall = nanmean(每个格点沿time算出来的RMSE图)

    返回：
    {
        "r_map": 原始R图,
        "rmse_map": 原始RMSE图,
        "R_mean": {
            "estimate": ...,
            "ci_lower": ...,
            "ci_upper": ...,
            "ci": (..., ...)
        },
        "RMSE_mean": {
            "estimate": ...,
            "ci_lower": ...,
            "ci_upper": ...,
            "ci": (..., ...)
        },
        "R_bootstrap": ...,
        "RMSE_bootstrap": ...
    }
    """
    predicty = np.asarray(predicty, dtype=np.float64)
    testy = np.asarray(testy, dtype=np.float64)

    if predicty.shape != testy.shape:
        raise ValueError(f"shape 不一致: {predicty.shape} vs {testy.shape}")
    if predicty.ndim != 3:
        raise ValueError("输入必须是三维 (time, lat, lon)")

    T = predicty.shape[0]
    rng = np.random.default_rng(random_state)
    alpha = (100 - ci) / 2

    # 原始点估计（与你旧代码一致）
    r_map, rmse_map = calc_r_rmse_maps_like_old(predicty, testy, show_progress=False)
    R_mean_est = np.nanmean(r_map)
    RMSE_mean_est = np.nanmean(rmse_map)

    # bootstrap
    R_boot = np.full(n_boot, np.nan, dtype=np.float64)
    RMSE_boot = np.full(n_boot, np.nan, dtype=np.float64)

    boot_iter = range(n_boot)
    if show_progress:
        boot_iter = tqdm(boot_iter, desc="Bootstrap")

    for b in boot_iter:
        idx = _sample_block_indices(T, block_size, rng)

        pred_b = predicty[idx, :, :]
        test_b = testy[idx, :, :]

        r_map_b, rmse_map_b = calc_r_rmse_maps_like_old(pred_b, test_b, show_progress=False)

        R_boot[b] = np.nanmean(r_map_b)
        RMSE_boot[b] = np.nanmean(rmse_map_b)

    # 去掉无效值
    R_boot_valid = R_boot[np.isfinite(R_boot)]
    RMSE_boot_valid = RMSE_boot[np.isfinite(RMSE_boot)]

    R_ci = (
        np.percentile(R_boot_valid, alpha),
        np.percentile(R_boot_valid, 100 - alpha)
    ) if len(R_boot_valid) > 0 else (np.nan, np.nan)

    RMSE_ci = (
        np.percentile(RMSE_boot_valid, alpha),
        np.percentile(RMSE_boot_valid, 100 - alpha)
    ) if len(RMSE_boot_valid) > 0 else (np.nan, np.nan)

    return {
        "r_map": r_map,
        "rmse_map": rmse_map,
        "R_mean": {
            "estimate": float(R_mean_est),
            "ci_lower": float(R_ci[0]),
            "ci_upper": float(R_ci[1]),
            "ci": (float(R_ci[0]), float(R_ci[1]))
        },
        "RMSE_mean": {
            "estimate": float(RMSE_mean_est),
            "ci_lower": float(RMSE_ci[0]),
            "ci_upper": float(RMSE_ci[1]),
            "ci": (float(RMSE_ci[0]), float(RMSE_ci[1]))
        },
        "R_bootstrap": R_boot_valid,
        "RMSE_bootstrap": RMSE_boot_valid
    }

In [3]:
import Auto_paint_self
cnn_transformer_predicty,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\cnn_transformer_pre_predicty_CN05_morev_sm2_lr000001.nc','predicty','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
cnn_transformer_testy,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\cnn_transformer_pre_testy_CN05_morev_sm2_lr000001.nc','testy','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\unet_pre_predicty_CN05_morev_sm2.nc','predicty','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\unet_pre_testy_CN05_morev_sm2.nc','testy','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transunet_predicty,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\transunet_pre_predicty_CN05_morev_sm2.nc','predicty','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transunet_testy,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\transunet_pre_testy_CN05_morev_sm2.nc','testy','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_predicty,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\transformer_pre_predicty_CN05_morev_sm2_lr0.001.nc','predicty','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transformer_testy,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\transformer_pre_testy_CN05_morev_sm2_lr0.001.nc','testy','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
lstm_predicty,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\lstm_pre_predicty_CN05_morev_sm2.nc','predicty','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
lstm_testy,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\lstm_pre_testy_CN05_morev_sm2.nc','testy','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_predicty_direct,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\unet_pre_predicty_CN05_morev_sm2_direct.nc','predicty','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
unet_testy_direct,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\unet_pre_testy_CN05_morev_sm2_direct.nc','testy','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transunet_predicty_direct,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\transunet_pre_predicty_CN05_morev_sm2_direct.nc','predicty','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
transunet_testy_direct,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\xuanxuan\transunet_pre_testy_CN05_morev_sm2_direct.nc','testy','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

F:\anaconda\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [4]:
CN05_PRE,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\CN05.1_Pre_1961_2021_daily_025x025.nc','pre','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
CMO_RPH,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\Multi-Sources_Precipitation_NC_2001_2022\CMORPH_China_2001_2022_2.nc','Pre','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
ERA5_PRE,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\Multi-Sources_Precipitation_NC_2001_2022\ERA5_China_2001_2022_2.nc','tp','yes','time','2017-10-22','2021-12-31','yes','longitude','yes','latitude',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
GPM_PRE,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\Multi-Sources_Precipitation_NC_2001_2022\GPM_China_2001_2022_time.nc','precipitation','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
GSMAP_PRE,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\Multi-Sources_Precipitation_NC_2001_2022\GSMAP_China_2001_2022_time.nc','Pre_daily','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
MSWEP_PRE,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\Multi-Sources_Precipitation_NC_2001_2022\MSWEP_China_2001_2022_2.nc','Pre','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
PERSIANN_CDR_PRE,lon,lat,levels,latlow,lattop,lonleft,lonright,times=Auto_paint_self.open_data_nc('one',r'E:\Multi-Sources_Precipitation_NC_2001_2022\PERSIANN_CDR_PRE_China_2001_2022_2.nc','Pre','yes','time','2017-10-22','2021-12-31','yes','lon','yes','lat',18.0,53.5,73.5,135.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')

C:\Users\TBYC\AppData\Roaming\Python\Python311\site-packages\xarray\coding\times.py:206: SerializationWarning: Ambiguous reference date string: 1-1-1 00:00:00. The first value is assumed to be the year hence will be padded with zeros to remove the ambiguity (the padded reference date string is: 0001-1-1 00:00:00). To remove this message, remove the ambiguity by padding your reference date strings with zeros.
  ref_date = _ensure_padded_year(ref_date)


In [5]:
import numpy as np
cnn_transformer_predicty=np.array(cnn_transformer_predicty)
unet_predicty=np.array(unet_predicty)
transunet_predicty=np.array(transunet_predicty)
transformer_predicty=np.array(transformer_predicty)
lstm_predicty=np.array(lstm_predicty)
unet_predicty_direct=np.array(unet_predicty_direct)
transunet_predicty_direct=np.array(transunet_predicty_direct)
cnn_transformer_predicty[cnn_transformer_predicty<0]=0
unet_predicty[unet_predicty<0]=0
transunet_predicty[transunet_predicty<0]=0
transformer_predicty[transformer_predicty<0]=0
lstm_predicty[lstm_predicty<0]=0
unet_predicty_direct[unet_predicty_direct<0]=0
transunet_predicty_direct[transunet_predicty_direct<0]=0

In [7]:
cnn_transformer_res = bootstrap_ci_like_old_r_rmse(cnn_transformer_predicty,cnn_transformer_testy,n_boot=1000,block_size=5,ci=95,random_state=42)

print(cnn_transformer_res["R_mean"]["ci"])
print(cnn_transformer_res["RMSE_mean"]["ci"])

Bootstrap:   4%|██▊                                                                | 42/1000 [11:35<4:24:49, 16.59s/it]F:\anaconda\Lib\site-packages\scipy\stats\_stats_py.py:4781: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
Bootstrap: 100%|█████████████████████████████████████████████████████████████████| 1000/1000 [4:40:25<00:00, 16.83s/it]

(0.3847264247465809, 0.4107616275812084)
(3.8701459444836455, 4.412881639481043)


In [8]:
unet_res = bootstrap_ci_like_old_r_rmse(unet_predicty,unet_testy,n_boot=1000,block_size=5,ci=95,random_state=42)

print(unet_res["R_mean"]["ci"])
print(unet_res["RMSE_mean"]["ci"])

Bootstrap: 100%|█████████████████████████████████████████████████████████████████| 1000/1000 [4:36:21<00:00, 16.58s/it]

(0.7248403219228977, 0.7413235240659373)
(2.6385388608100886, 3.0150574980676677)


In [9]:
transunet_res = bootstrap_ci_like_old_r_rmse(transunet_predicty,transunet_testy,n_boot=1000,block_size=5,ci=95,random_state=42)

print(transunet_res["R_mean"]["ci"])
print(transunet_res["RMSE_mean"]["ci"])

Bootstrap: 100%|█████████████████████████████████████████████████████████████████| 1000/1000 [4:35:38<00:00, 16.54s/it]

(0.7427904900457938, 0.7634969942062627)
(2.4993392969521877, 2.8645231969485305)


In [10]:
transformer_res = bootstrap_ci_like_old_r_rmse(transformer_predicty,transformer_testy,n_boot=1000,block_size=5,ci=95,random_state=42)

print(transformer_res["R_mean"]["ci"])
print(transformer_res["RMSE_mean"]["ci"])

Bootstrap: 100%|█████████████████████████████████████████████████████████████████| 1000/1000 [4:39:11<00:00, 16.75s/it]

(0.6522754983743878, 0.6698473497189017)
(2.80910723418378, 3.2062010815494477)


In [11]:
lstm_res = bootstrap_ci_like_old_r_rmse(lstm_predicty,lstm_testy,n_boot=1000,block_size=5,ci=95,random_state=42)

print(lstm_res["R_mean"]["ci"])
print(lstm_res["RMSE_mean"]["ci"])

Bootstrap: 100%|█████████████████████████████████████████████████████████████████| 1000/1000 [4:43:09<00:00, 16.99s/it]

(0.6186316212302974, 0.6383440869333584)
(18.537412222097288, 21.04479425059661)


In [12]:
transunet_res_direct = bootstrap_ci_like_old_r_rmse(transunet_predicty_direct,transunet_testy_direct,n_boot=1000,block_size=5,ci=95,random_state=42)

print(transunet_res_direct["R_mean"]["ci"])
print(transunet_res_direct["RMSE_mean"]["ci"])

Bootstrap: 100%|█████████████████████████████████████████████████████████████████| 1000/1000 [6:06:35<00:00, 22.00s/it]

(0.6963867760131135, 0.714469987917311)
(2.729611954660274, 3.116698459804505)
